In [ ]:
import networkx as nx
import numpy as np
import random

def generate_random_repo_graph(num_files, avg_dependencies, size_scale=100):
    """
    Generates a random graph modeling a code repository.

    Parameters:
    - num_files (int): Number of files in the repository.
    - avg_dependencies (float): Average number of dependencies per file.
    - size_scale (int): Scale factor for file sizes.

    Returns:
    - G (networkx.DiGraph): A directed graph where nodes represent files and edges represent imports.
    - file_sizes (dict): A dictionary mapping each node to its file size (number of tokens)
    """
    # Step 1: Create a directed graph
    G = nx.DiGraph()

    # Step 2: Assign file sizes using a log-normal distribution
    # Log-normal parameters chosen to simulate heavy-tailed distributions
    mu, sigma = 3, 1  # Mean and standard deviation for log-normal
    file_sizes = {i: int(np.random.lognormal(mu, sigma) * size_scale) for i in range(num_files)}

    # Step 3: Add nodes with file size attributes
    for node, size in file_sizes.items():
        G.add_node(node, size=size)

    # Step 4: Add edges based on dependencies
    for node in range(num_files):
        num_deps = np.random.poisson(avg_dependencies)  # Number of dependencies follows Poisson distribution
        potential_targets = list(set(range(num_files)) - {node})  # Avoid self-loops
        targets = random.sample(potential_targets, min(num_deps, len(potential_targets)))
        for target in targets:
            G.add_edge(node, target)

    return G, file_sizes

# Example usage:
num_files = 50  # Number of files in the repository
avg_dependencies = 5  # Average number of dependencies per file
graph, sizes = generate_random_repo_graph(num_files, avg_dependencies)

# Print summary
print(f"Generated graph with {len(graph.nodes)} nodes and {len(graph.edges)} edges.")
print("Sample node sizes:", list(sizes.items())[:5])

Generated graph with 50 nodes and 239 edges.
Sample node sizes: [(0, 4007), (1, 2305), (2, 793), (3, 5339), (4, 1708)]


In [ ]:
import numpy as np

def generate_error_pdf(graph, file_sizes, error_model='linear'):
    """
    Generates a probability distribution function (PDF) for nodes in the graph based on file sizes
    and an error probability model.

    Parameters:
    - graph (networkx.DiGraph): The directed graph representing the code repository.
    - file_sizes (dict): A dictionary mapping each node to its file size.
    - error_model (str): The model to use for error probability ('linear', 'quadratic', etc.).

    Returns:
    - pdf (dict): A dictionary mapping each node to its probability of being selected.
    """
    # Step 1: Apply the error model to file sizes
    if error_model == 'linear':
        error_weights = {node: size for node, size in file_sizes.items()}
    elif error_model == 'quadratic':
        error_weights = {node: size**2 for node, size in file_sizes.items()}
    elif error_model == 'logarithmic':
        # Logarithmic scaling to reduce the impact of very large files
        error_weights = {node: np.log(size + 1) for node, size in file_sizes.items()}
    else:
        raise ValueError("Unsupported error model")

    # Step 2: Normalize weights to create a probability distribution
    total_weight = sum(error_weights.values())
    pdf = {node: weight / total_weight for node, weight in error_weights.items()}

    return pdf

# Example usage with dummy data
dummy_graph = None  # Placeholder for the graph object
dummy_file_sizes = {0: 100, 1: 200, 2: 300, 3: 400, 4: 500}  # Example file sizes

# Generate PDFs with different models
error_pdf_linear = generate_error_pdf(dummy_graph, dummy_file_sizes, error_model='linear')
error_pdf_quadratic = generate_error_pdf(dummy_graph, dummy_file_sizes, error_model='quadratic')
error_pdf_logarithmic = generate_error_pdf(dummy_graph, dummy_file_sizes, error_model='logarithmic')

print("Linear Model PDF:", error_pdf_linear)
print("Quadratic Model PDF:", error_pdf_quadratic)
print("Logarithmic Model PDF:", error_pdf_logarithmic)

In [ ]:
CACHE_SIZE = 5
MAX_NODES = 100
MIN_NODES = 12
ITERS_PER_GRAPH = 30
ITERS_PER_SIZE = 30

## PageRank + LRU

In [ ]:
import networkx as nx
import random
from tqdm import tqdm
from collections import defaultdict, OrderedDict

page_rank_lru_hit_rate = defaultdict(int) # hit_rate[i] = hit rate for graph size i

for n in tqdm(range(MIN_NODES, MAX_NODES+1)):  # Loop over graph sizes
    hit_rate = []
    
    for _ in range(ITERS_PER_SIZE):  # Generate random graphs for each size
        total_hits = 0
        total_misses = 0
        p = random.uniform(0, 1)  # Randomly select edge probability
        G = random_graph(n, p)
        frequency = defaultdict(int)
        page_rank = nx.pagerank(G)
        # Cache the top 10 nodes based on PageRank
        cached_nodes = set(sorted(page_rank, key=page_rank.get, reverse=True)[:CACHE_SIZE//2])
        lru_cache_size = CACHE_SIZE - len(cached_nodes)
        lru_cache = OrderedDict()
        init = set(random.choices(list(set(G.nodes()) - cached_nodes), k=lru_cache_size))
        for node in init:
            lru_cache[node] = None
        assert set(lru_cache.keys()) & cached_nodes == set()
        for _ in range(ITERS_PER_GRAPH):  # Randomly select nodes to test
            assert set(lru_cache.keys()) & cached_nodes == set()
            u = random.choices(list(G.nodes()), weights=[G.nodes[node]['weight'] for node in G.nodes()])[0]
            # Get dependencies of `u`
            dependencies = set(G[u]) | {u}

            # Update cache based on dependencies
            for dep in dependencies:
                if dep in cached_nodes: continue
                if dep in lru_cache:
                    # Move the accessed item to the end (mark as recently used)
                    lru_cache.move_to_end(dep)
                else:
                    # Add new item, evict least recently used if cache is full
                    if len(lru_cache) >= lru_cache_size:
                        lru_cache.popitem(last=False)  # Remove the first (LRU) item
                    lru_cache[dep] = None  # Add the new dependency to the cache
            
            # Count how many of `u`'s dependencies are in the cache
            num_hits = len(dependencies & cached_nodes) + len(dependencies & set(lru_cache.keys()))
            total_hits += num_hits
            
            # Count the number of dependencies not in the cache as misses
            total_misses += len(dependencies) - num_hits
        hit_rate.append(total_hits / (total_hits + total_misses) if total_hits + total_misses > 0 else 0)
    
    # Calculate and store hit rate for current graph size
    page_rank_lru_hit_rate[n] = np.mean(hit_rate)